# Ordered Logistic Regression Results for Adoption Predictors: FAIR^2 dataset Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and accessed programmatically.

In [ ]:
# If you have not yet installed mlcroissant, uncomment and run the next line:
!pip install mlcroissant

## 1. Data Loading
Load the FAIR^2 dataset's metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their `@id` fields. These uniquely identify each table (or logical record set) in the Croissant package.

In [ ]:
# List out all available record sets by @id, name, and description
record_sets = list(dataset.record_sets)
print(f"Total Record Sets: {len(record_sets)}")
for i, rs in enumerate(record_sets):
    print(f"[{i}] Record Set @id: {rs.id}")
    print(f"    name: {rs.name if hasattr(rs, 'name') else ''}")
    print(f"    description: {rs.description if hasattr(rs, 'description') else ''}\n")

# For demonstration, print the available fields and columns for the first record set (if available)
if record_sets:
    rs = record_sets[0]
    # Fields
    print(f"Fields in Record Set (@id = {rs.id}):")
    for field in rs.fields:
        print(f"  Field @id: {field.id}, name: {field.name if hasattr(field, 'name') else ''}, data type: {field.data_type if hasattr(field, 'data_type') else ''}")
        # Columns, if present
        if hasattr(field, 'columns') and field.columns is not None:
            for col in field.columns:
                print(f"    Column @id: {col.id}, name: {col.name if hasattr(col, 'name') else ''}")

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis.

**All record sets, fields, and columns are referenced by their `@id` values for maximum clarity.**

In [ ]:
# Build a list of all record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records for record set '{rs.name if hasattr(rs, 'name') else rs.id}': @id={rs.id}")

# Preview the columns and first rows from each Record Set (by @id)
for rs_id, df in dataframes.items():
    print(f"\nRecord Set @id: {rs_id}\nColumns: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's practice EDA on a numeric field from a selected record set, referencing by `@id`.

We'll demonstrate filtering, normalization, and group-wise aggregation. **Please replace variable placeholders with correct `@id` values as needed for your analysis.**

In [ ]:
# --- Configuration: Select Record Set and Field by @id ---
# Replace these with valid IDs according to your dataset overview!
target_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(target_record_set_id)

# Auto-select a numeric column if possible
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    print('No numeric field found in the selected record set for demonstration.')
else:
    print(f"Numeric field chosen (by @id): {numeric_field_id}")

    # Example threshold; adjust as needed
    threshold = df[numeric_field_id].mean() if pd.notna(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f} (mean):\n{filtered_df.head()}")

    # Normalization (z-score)
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to group by a non-numeric field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
            group_field_id = col
            break
    if group_field_id:
        print(f"\nGrouping records by '{group_field_id}' (@id)...")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships. All references use `@id` fields. Below is a histogram for the numeric field and a boxplot for groups if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

if group_field_id:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- In this notebook, we loaded and explored the FAIR^2 dataset, referencing all elements by their `@id` fields, consistent with the Croissant and mlcroissant design.
- The workflow included reviewing metadata, record sets, and field IDs, and extracting data for basic EDA and visualization.
- Adjust the selected record set and field `@id`s in the notebook for deeper, field-specific analyses suited to your research or modeling goals.